In [ ]:
# This script takes your DICOM data (4D Flow MRI) and transforms them into 4D arrays that will be stored in the h5 file
# Make sure you have DicomData and h5functions when you run this script
# This script is taken from 4DFlowNet by Ferdinan et al. (2020) and modified accordingly
# Be carefull when defining the DICOM data Path, you must have your DICOM data in three different files, for each enconding direction (RL,FH,AP) for both Magnitude and Phase
# Input directory contains the Dicom files with [Phase/Magnitude series desc]/[triggerTime] structure'
# The output directory should be in the same file as the your three DICOM data files
import numpy as np
import pydicom
import os
import argparse
from DicomData import DicomData

def get_filepaths(directory):
    """
        This function will generate the file names in a directory 
        tree by walking the tree either top-down or bottom-up. For each 
        directory in the tree rooted at directory top (including top itself), 
        it yields a 3-tuple (dirpath, dirnames, filenames).
    """
    file_paths = []  # List which will store all of the full filepaths.

    # Walk the tree.
    for root, directories, files in os.walk(directory):
        for filename in files:
            # Join the two strings in order to form the full filepath.
            filepath = os.path.join(root, filename)
            file_paths.append(filepath)  # Add it to the list.

    return file_paths 

def get_volume(vol_dir):
    """
        Get phase or magnitude image volume
    """
    volume = []
    # Retrieve all the dicom filepaths
    files  = get_filepaths(vol_dir)
    
    for slice_nr, dicom_path in enumerate(files):
        ds = pydicom.dcmread(dicom_path)
        img = ds.pixel_array
        
        if slice_nr == 0:
            # Get this on the first slice only
            spacing = ds.PixelSpacing
            spacing.append(ds.SliceThickness)
            spacing = np.asarray(spacing)
            
            # Note: In our case, sequence name contains venc and direction info
            sequence_name = ds.SequenceName
            # print(sequence_name)

        volume.append(img)
    volume = np.asarray(volume)
    return volume, spacing, sequence_name

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-dir', type=str, required=True, help='Input directory contains the Dicom files with [Phase/Magnitude series desc]/[triggerTime] structure')
    parser.add_argument('--output-dir', type=str, default=r"the path where you 3 Magnitude&Phase files are stored ", help='Output directory where the dataset will be saved')
    parser.add_argument('--output-filename', type=str, default='mri_data_check.h5', help='Output filename (HDF5)')
    parser.add_argument('--phase-pattern', type=str, default='_P_', help='Pattern of phase series description')
    parser.add_argument('--mag-pattern', type=str, default='_M_', help='Pattern of magnitude series description')
    parser.add_argument('--fh-mul', type=int, default= -1, help='Velocity multiplier for Foot-Head direction')
    parser.add_argument('--rl-mul', type=int, default=  1, help='Velocity multiplier for Right-Left direction')
    parser.add_argument('--in-mul', type=int, default=  1, help='Velocity multiplier for Inplane direction')
    args = parser.parse_args()
    
    case_dir      = args.input_dir
    phase_pattern = args.phase_pattern
    mag_pattern   = args.mag_pattern
    output_path   = args.output_dir
    output_filename = args.output_filename
    in_multiplier = args.in_mul
    fh_multiplier = args.fh_mul
    rl_multiplier = args.rl_mul
    
    output_filepath = f'{output_path}/{output_filename}'

    # 1. Get the phase and magnitude directories 
    directories = os.listdir(case_dir)
    phase_dirs = [item for item in directories if phase_pattern in item]
    mag_dirs   = [item for item in directories if mag_pattern   in item]
    print("Phase dirs:\n", "\n".join(phase_dirs))
    print("Mag dirs:\n"  , "\n".join(mag_dirs))

    assert len(phase_dirs) == 3, f"There must be exactly 3 Phase directories with {phase_pattern} pattern"
    assert len(mag_dirs)   == 3, f"There must be exactly 3 Magnitude directories with {mag_pattern} pattern"

    # 2. Get and sort the trigger times
    dirpath = f'{case_dir}/{phase_dirs[0]}'
    timeFrames = os.listdir(dirpath)
    timeFrames = sorted(timeFrames, key=float)
    # print('All frames sorted:', timeFrames)

    # Create the output dir
    if not os.path.isdir(output_path):
        os.makedirs(output_path)

    # 3. Looping through the time frames
    for j in range(0, len(timeFrames)):
        triggerTime = timeFrames[j]
        print(f"\rProcessing {j+1}/{len(timeFrames)} (frame {triggerTime})        ", end="\r")

        # Wrap it as DicomData instance
        dicom_data = DicomData()
        # Collect the 3 phase and magnitude volumes for 1 time frame
        for mag_dir, p_dir in zip(mag_dirs, phase_dirs):
            magnitude_path = f'{case_dir}/{mag_dir}/{triggerTime}'
            phase_path = f'{case_dir}/{p_dir}/{triggerTime}'

            # Get the magnitude and phase images
            mag_images, _, _                = get_volume(magnitude_path)
            phase_images, spacing, sequence = get_volume(phase_path)
            
            dicom_data._phaseImages.append(phase_images)
            dicom_data._magImages.append(mag_images)
            dicom_data.sequenceNames.append(sequence)
            dicom_data.spacing = spacing
        # Save per row
        dicom_data.determine_velocity_components(in_multiplier, fh_multiplier, rl_multiplier)
        dicom_data.save_dataset(output_filepath, triggerTime)

    # End of trigger time loop
    print(f'\nDone! saved at {output_filepath}')


if __name__ == '__main__':
    main()

In [ ]:
#The following are scripts that evaluate various DICOM tags 
import pydicom
import os

def check_dicom_sequence_name(dicom_dir):
    # Get the file paths of DICOM files in the directory
    dicom_files = [f for f in os.listdir(dicom_dir) if f.endswith('.dcm')]

    for dicom_file in dicom_files:
        dicom_path = os.path.join(dicom_dir, dicom_file)
        
        # Read the DICOM file
        ds = pydicom.dcmread(dicom_path)

        # Print the file name
        print(f"\nDICOM File: {dicom_file}")
        
        # Print the SequenceName if it exists
        sequence_name = getattr(ds, 'SequenceName', 'SequenceName not found')
        print(f"SequenceName: {sequence_name}")

        # Optionally, print more attributes for inspection
        print(f"Modality: {ds.Modality}")
        print(f"Patient Name: {ds.PatientName}")
        print(f"Study Date: {ds.StudyDate}")
        print(f"Series Description: {ds.SeriesDescription}")
        print(f"Image Position Patient: {ds.ImagePositionPatient}")

# Usage: pass your DICOM directory
dicom_directory = r"the path where you 3 Magnitude&Phase files are stored\Magnitude_AP_M_\0"
check_dicom_sequence_name(dicom_directory)

In [ ]:
import pydicom
import os

def inspect_dicom_venc_and_velocity(dicom_dir):
    # Get the list of DICOM files in the directory
    dicom_files = [f for f in os.listdir(dicom_dir) if f.endswith('.dcm')]

    for dicom_file in dicom_files:
        dicom_path = os.path.join(dicom_dir, dicom_file)
        
        # Read the DICOM file
        ds = pydicom.dcmread(dicom_path)
        
        print(f"\nInspecting DICOM file: {dicom_file}")
        
        # Check for SequenceName, which might contain VENC and direction
        sequence_name = getattr(ds, 'SequenceName', 'Not found')
        print(f"SequenceName: {sequence_name}")
        
        # Search through all DICOM tags for VENC or velocity-related information
        for elem in ds:
            # Look for any tag that might relate to VENC or velocity
            if 'venc' in str(elem).lower() or 'velocity' in str(elem).lower() or 'direction' in str(elem).lower():
                print(f"Potential VENC/Velocity info: {elem}")

        # Optionally, print a few more known fields that might contain relevant information
        # You can add or remove fields based on your DICOM data structure
        print(f"Modality: {ds.Modality}")
        print(f"Series Description: {getattr(ds, 'SeriesDescription', 'Not found')}")
        print(f"Study Description: {getattr(ds, 'StudyDescription', 'Not found')}")

        # Print all pixel-related information for deeper insights
        print(f"Pixel Spacing: {getattr(ds, 'PixelSpacing', 'Not found')}")
        print(f"Image Position Patient: {getattr(ds, 'ImagePositionPatient', 'Not found')}")
        print(f"Image Orientation Patient: {getattr(ds, 'ImageOrientationPatient', 'Not found')}")
        
        # Stop after a few files to avoid too much output (you can adjust this)
        if dicom_files.index(dicom_file) >= 2:
            break

# Usage: Set your DICOM directory here
dicom_directory = r"the path where you 3 Magnitude&Phase files are stored\Phase_RL_P_\0"
inspect_dicom_venc_and_velocity(dicom_directory)

In [ ]:
def read_venc_from_dicom(dicom_path):
    ds = pydicom.dcmread(dicom_path)

    # Get PC Velocity field (2001, 101a)
    pc_velocity = ds.get((0x2001, 0x101a), None)
    if pc_velocity:
        venc = pc_velocity.value  # Get the VENC value
        print(f"VENC found: {venc}")
    else:
        print("PC Velocity (VENC) not found")

    # Use SeriesDescription to get direction (e.g., AP, RL, FH)
    series_description = ds.SeriesDescription
    if "AP" in series_description:
        direction = "Anterior-Posterior (AP)"
    elif "RL" in series_description:
        direction = "Right-Left (RL)"
    elif "FH" in series_description:
        direction = "Foot-Head (FH)"
    else:
        direction = "Unknown direction"

    print(f"Velocity Direction: {direction}")

# Example usage
dicom_file = r"the path where you 3 Magnitude&Phase files are stored\Magnitude_AP_M_\0\Mag (0001).dcm"
read_venc_from_dicom(dicom_file)

In [ ]:
import pydicom
import os

def get_filepaths(directory):
    """
    Generate the file names in a directory tree.
    """
    file_paths = []
    for root, _, files in os.walk(directory):
        for filename in files:
            if filename.lower().endswith('.dcm'):  # Ensure we are only reading DICOM files
                filepath = os.path.join(root, filename)
                file_paths.append(filepath)
    return file_paths

def read_dicom_tags(directory):
    # Get all DICOM file paths in the directory
    dicom_files = get_filepaths(directory)
    
    for dicom_file in dicom_files:
        print(f"Inspecting {dicom_file}")
        ds = pydicom.dcmread(dicom_file)

        # Print relevant tags
        print("DICOM Tags and Values:")
        for elem in ds:
            # elem is a DataElement object; access its properties
            tag = elem.tag
            description = elem.name
            value = elem.value
            
            print(f"Tag: {tag}, Description: {description}, Value: {value}")
        print("\n")

if __name__ == '__main__':
    # Specify the directory containing your DICOM files
    dicom_directory = 'C:-----\\StudyDir_11102024034642\\ST1\\S2'
    read_dicom_tags(dicom_directory)

In [ ]:
import os
import pydicom

# Define the main folder and the attribute details
main_folder = r"the path where you 3 Magnitude&Phase files are stored\Phase_RL_P_"
attribute_group = (0x0018, 0x0024)  # (Group, Element)
attribute_value = "fl4d1_v180rl"

# Function to add an attribute to DICOM files
def add_dicom_attribute(folder):
    for dirpath, _, files in os.walk(folder):
        for file in files:
            if file.lower().endswith('.dcm'):  # Check for DICOM files
                file_path = os.path.join(dirpath, file)
                try:
                    # Read the DICOM file
                    ds = pydicom.dcmread(file_path)
                    
                    # Add the new attribute
                    ds.add_new(attribute_group, 'LO', attribute_value)  # 'LO' for Long String
                    
                    # Save the modified DICOM file
                    ds.save_as(file_path)
                    print(f"Updated {file_path} with Sequence Name: {attribute_value}")
                except Exception as e:
                    print(f"Failed to process {file_path}: {e}")

# Call the function
add_dicom_attribute(main_folder)